# Commodity forecasting | Feature research

**Completion gate: OPEN.** This notebook reports the first controlled feature-family experiment. It does not claim that feature engineering is exhausted or that the final model is selected.

The initial comparison holds the algorithm and regularization fixed: Ridge regression with $\alpha=100$. Only the representation changes. This isolates a first estimate of feature value before stronger model optimization.

In [ ]:
from pathlib import Path
import sys
if not Path("pyproject.toml").exists():
    sys.path.insert(0, str(Path.cwd().parent))
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
from scripts.notebook_support import project_root, checked_reports, show_figure
root = project_root()
audit, research = checked_reports(root)
config = json.loads((root / "configs/research.json").read_text())
display(Markdown(f"**Verified experiment:** `{research['lineage'][:16]}` · **Feature gate:** open"))

## Candidate families and hypotheses

Returns and momentum capture persistence; reversion measures departures from recent levels; volatility separates movement size from direction. Liquidity and OHLC structure describe trading activity and intraday shape. Market-relative features describe shared context, while target-aligned pair spreads use the competition's economic relationships. All inputs are observable at or before the prediction row.

In [ ]:
family_counts = pd.Series(research["family_counts"]).sort_values().rename_axis("Family").reset_index(name="Candidates")
fig = px.bar(family_counts, x="Candidates", y="Family", orientation="h", text="Candidates", title=f"{research['candidate_count']:,} domain-motivated candidates across nine families")
fig.update_traces(marker_color="#1F6C99", textposition="outside")
show_figure(fig, root, "feature_families", 570)

## Fit the screener only on training dates

For each fold and each ablation: reject over-40%-missing and constant features; remove exact duplicates; rank features by mean absolute target correlation using only observed training labels; remove highly correlated selected candidates; cap the diagnostic model at 96 features. Training medians, means, and scales are reused unchanged for validation.

The count is a candidate search budget, not a claim that thousands of independent signals exist. Final retained features remain undecided.

In [ ]:
lineage = research["lineage"]
features_dir = root / "artifacts" / lineage / "features"
from commodity_prediction.runtime import verify_checkpoint
assert verify_checkpoint(features_dir, lineage)
families = json.loads((features_dir / "families.json").read_text())
last_fold = research["folds_completed"] - 1
screen = pd.read_csv(root / "artifacts" / lineage / f"fold_{last_fold}" / "all_families" / "screening.csv")
decision = screen.assign(reason=screen.reason.str.split(":").str[0]).groupby("reason").size().sort_values(ascending=False)
display(decision.to_frame("Features"))
assert len(screen) == research["candidate_count"]
assert int((screen.status == "retained").sum()) <= config["max_features"]

## Official metric and additive ablations

The official metric is the mean daily cross-sectional Spearman rank correlation divided by its **population** standard deviation. Higher is better; no annualization factor is applied. The code is parity-tested against an independent Spearman calculation, including ties and missing labels.

[Official metric](https://www.kaggle.com/code/metric/mitsui-co-commodity-prediction-metric). Each family is added separately to the same one-date-return reference; an additional experiment combines all families. These are historical validation scores, not Kaggle submissions.

In [ ]:
comparison = pd.DataFrame(research["comparison"])
display(comparison[["variant", "official_metric", "delta_from_reference", "fold_metrics", "retained_per_fold"]].round(4))
ordered = comparison.sort_values("official_metric")
fig = px.bar(ordered, x="official_metric", y="variant", orientation="h", text_auto=".3f", title="Which representations improve the fixed diagnostic model?")
fig.update_traces(marker_color="#27A394", textposition="outside")
fig.update_layout(xaxis_title="Pooled walk-forward correlation Sharpe", yaxis_title="Feature experiment")
show_figure(fig, root, "ablation_scores", 600)

In [ ]:
rows = []
for result in research["results"]:
    rows.append({"Fold": f"Fold {result['fold'] + 1}", "Variant": result["variant"], "Metric": result["official_metric"]})
matrix = pd.DataFrame(rows).pivot(index="Variant", columns="Fold", values="Metric")
fig = px.imshow(matrix, text_auto=".2f", color_continuous_scale="RdBu", color_continuous_midpoint=0,
                aspect="auto", title="Average gains can hide unstable periods", labels={"color": "Metric"})
show_figure(fig, root, "fold_stability", 590)

In [ ]:
delta = comparison.loc[comparison.variant != "reference"].sort_values("delta_from_reference").copy()
lower = delta.conditional_delta_95_interval.map(lambda v: v[0])
upper = delta.conditional_delta_95_interval.map(lambda v: v[1])
fig = go.Figure(go.Scatter(x=delta.delta_from_reference, y=delta.variant, mode="markers",
    marker={"size": 11, "color": "#1F6C99"},
    error_x={"type": "data", "symmetric": False, "array": upper - delta.delta_from_reference,
             "arrayminus": delta.delta_from_reference - lower}))
fig.add_vline(x=0, line_dash="dash", line_color="#9EAFBF")
fig.update_layout(title="Uncertainty around the initial feature gains", xaxis_title="Metric change versus reference · conditional 95% interval")
show_figure(fig, root, "ablation_uncertainty", 570)

The paired circular block bootstrap uses 20-date blocks and 500 resamples. It conditions on already fitted models, does not refit the screener, and is not adjusted for testing multiple families. It is an initial uncertainty diagnostic rather than a definitive significance claim.

In [ ]:
best = research["comparison"][0]
display(Markdown(f"**Initial leader:** `{best['variant']}` · pooled official metric **{best['official_metric']:.4f}** · change versus reference **{best['delta_from_reference']:+.4f}**.\n\nCompleted **{research['experiments_completed']} fold/representation experiments** across all 424 targets. The final 252 dates remain unused for model selection."))
display(pd.DataFrame({"Open research work": ["Conditional drop-family ablations and group permutation importance", "Nonlinear model controls without broad tuning", "Longer-window regimes, nonlinear interactions, and stability selection", "Target/pair-specific screening versus shared screening", "Release-aware pooling or target encoding only where it adds information", "External point-in-time data assessment and source vintages", "Block-bootstrap sensitivity and model-selection uncertainty", "Locked final feature gate, then final model comparison and holdout"]}))

## Decision

Keep the feature gate open. This experiment establishes a reproducible reference and empirical family comparisons. It does not yet establish diminishing returns, a production model, a trading strategy, or an employer-facing project completion score. Follow-up experiments must preserve this lineage, reuse verified stages, and document negative results as carefully as positive ones.